# Notebook 2 — The Annuity Agent

**Companion notebook to:** *From Exam Question to Autonomous Agent: Teaching Agentic AI Through Financial
Mathematics* (CAS Global Teaching Materials Innovation Challenge submission)

This notebook builds and runs the agent described in **Section 4.2** of the case study: an agent that computes
the present and accumulated values of a level annuity, and demonstrates why the annuity-immediate versus
annuity-due timing convention matters just as much as the rate convention did in Notebook 1.

**Before running:** get a free Gemini API key at https://aistudio.google.com/app/apikey and paste it into the
`GEMINI_API_KEY` cell below (or set it as a Colab secret named `GEMINI_API_KEY`).

## 0. Setup

This notebook runs unchanged in **Google Colab** or in a **local editor** (VS Code, PyCharm, JupyterLab) using the
`uv`-managed environment that ships alongside these notebooks (`pyproject.toml` + `uv.lock`). The cell below
detects which one it is running in and does the right thing automatically:

- **Colab**: installs the required packages directly into the Colab runtime (nothing to download beforehand).
- **Local**: assumes you already ran `uv sync` once in the project folder (see `README.md`), so the packages are
  already present in `.venv` — this cell skips installation and just confirms the imports work.

Either way, run this cell once per session.

In [ ]:
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "agno[google,opentelemetry,sqlite]", "google-genai",
            "openinference-instrumentation-agno", "python-dotenv",
        ],
        check=True,
    )
else:
    print(
        "Running outside Colab -- assuming packages were already installed via "
        "`uv sync` in this project's folder (see README.md). Skipping pip install."
    )

import agno, google.genai, dotenv  # noqa: F401 -- import check only
print("Environment ready.")

In [ ]:
import os

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Option A (recommended): click the key icon in Colab's left sidebar, add a
    # secret named GEMINI_API_KEY, and this line picks it up automatically.
    try:
        from google.colab import userdata
        os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    except Exception:
        pass
    # Option B: no Colab secret found -- paste your key directly here instead.
    if not os.environ.get("GEMINI_API_KEY"):
        os.environ["GEMINI_API_KEY"] = "PASTE_YOUR_GEMINI_API_KEY_HERE"
else:
    # Local: reads GEMINI_API_KEY from a ".env" file in the project root.
    # Copy .env.example to .env and fill it in once -- see README.md.
    from dotenv import load_dotenv
    load_dotenv()

assert os.environ.get("GEMINI_API_KEY") and "PASTE_YOUR" not in os.environ["GEMINI_API_KEY"], (
    "GEMINI_API_KEY is not set. In Colab: add a secret named GEMINI_API_KEY, or "
    "paste your key directly into this cell. Locally: copy .env.example to .env "
    "in the project folder and add your key there."
)
print("GEMINI_API_KEY is set.")

## 1. The Financial Mathematics and the math (Section 4.2)

An annuity is a series of level payments. Paid at the **end** of each period (annuity-immediate) for `n` periods
at effective rate `i` per period, its present value has the closed form:

$$a_{\overline{n}|} = \frac{1 - v^n}{i}, \qquad v = \frac{1}{1+i}$$

Paid instead at the **start** of each period (annuity-due), every payment is worth exactly `(1 + i)` times as
much today:

$$\ddot a_{\overline{n}|} = (1+i) \times a_{\overline{n}|}$$

Its accumulated value at the end of the term follows the same relationship projected forward rather than
discounted back.

Whether a request describes an annuity-immediate or an annuity-due is a *timing* detail, and everyday language is
often silent about it. "I'll deposit $500 a year for 10 years starting today" is unambiguously a due; "I'll
deposit $500 a year for 10 years" alone is not.

## 2. Python implementation (Section 4.2)

In [ ]:
from agno.exceptions import RetryAgentRun


def accumulated_value(principal: float, annual_rate: float, years: float) -> float:
    """
    Compute the accumulated (future) value of a lump-sum investment
    under compound interest.

    Args:
        principal (float): The amount invested today. Must be
            non-negative.
        annual_rate (float): The effective annual interest rate, as a
            decimal (e.g., 0.06 for 6%). Must be greater than -1.
        years (float): The number of years the amount is invested.
            Must be non-negative.

    Returns:
        float: The accumulated value after the given number of years.
    """
    if principal < 0:
        raise RetryAgentRun("principal must be non-negative. Re-check the request.")
    if years < 0:
        raise RetryAgentRun("years must be non-negative. Re-check the request.")
    if annual_rate <= -1:
        raise RetryAgentRun("annual_rate must be greater than -100%. Re-check the request.")
    return principal * (1 + annual_rate) ** years


def annuity_immediate_pv(payment: float, rate_per_period: float, n: int) -> float:
    """
    Compute the present value of a level annuity-immediate.

    Args:
        payment (float): The level payment amount per period. Must be
            non-negative.
        rate_per_period (float): The effective interest rate per
            payment period, as a decimal. Must be greater than -1.
        n (int): The number of payment periods. Must be positive.

    Returns:
        float: The present value of the annuity.
    """
    if payment < 0:
        raise RetryAgentRun("payment must be non-negative. Re-check the request.")
    if n <= 0:
        raise RetryAgentRun("n must be a positive number of periods. Re-check the request.")
    if rate_per_period <= -1:
        raise RetryAgentRun("rate_per_period must be greater than -100%. Re-check the request.")
    if rate_per_period == 0:
        return payment * n
    v = 1 / (1 + rate_per_period)
    a_n = (1 - v ** n) / rate_per_period
    return payment * a_n


def annuity_due_pv(payment: float, rate_per_period: float, n: int) -> float:
    """
    Compute the present value of a level annuity-due, by composing
    the annuity-immediate calculation.

    Args:
        payment (float): The level payment amount per period. Must be
            non-negative.
        rate_per_period (float): The effective interest rate per
            payment period, as a decimal. Must be greater than -1.
        n (int): The number of payment periods. Must be positive.

    Returns:
        float: The present value of the annuity-due.
    """
    return (1 + rate_per_period) * annuity_immediate_pv(payment, rate_per_period, n)


def annuity_immediate_av(payment: float, rate_per_period: float, n: int) -> float:
    """
    Compute the accumulated value of a level annuity-immediate, by
    composing its present value with ordinary compounding.

    Args:
        payment (float): The level payment amount per period. Must be
            non-negative.
        rate_per_period (float): The effective interest rate per
            payment period, as a decimal. Must be greater than -1.
        n (int): The number of payment periods. Must be positive.

    Returns:
        float: The accumulated value of the annuity.
    """
    pv = annuity_immediate_pv(payment, rate_per_period, n)
    return accumulated_value(pv, rate_per_period, n)  # from the same Time Value of Money family as Section 4.1


# Sanity check against the case study's own worked numbers (Section 4.2):
print(f"annuity-due PV:       {annuity_due_pv(500, 0.06, 10):.2f}")
print(f"annuity-immediate PV: {annuity_immediate_pv(500, 0.06, 10):.2f}")

Note what `annuity_immediate_av` does *not* do: it does not re-derive the accumulated-value summation from
scratch. It calls a present-value tool that is already tested, then calls an already-tested compounding tool.
Composing verified tools, rather than re-implementing a formula a second time, is a design habit worth teaching
explicitly, not just an implementation shortcut.

## 3. Agentic integration (Section 4.2)

In [ ]:
from agno.agent import Agent
from agno.models.google import Gemini

annuity_agent = Agent(
    name="Annuity Agent",
    role="Computes present and accumulated values of level annuities.",
    model=Gemini(id="gemini-3.5-flash", temperature=0.0),
    tools=[annuity_immediate_pv, annuity_due_pv, annuity_immediate_av],
    instructions=[
        "Always use a tool to perform the calculation. Never state a "
        "present or accumulated value that did not come from a tool "
        "call.",
        "If a request does not specify whether payments occur at the "
        "start or end of each period, ask a brief clarifying question "
        "rather than assuming an annuity-immediate.",
    ],
    markdown=True,
)

annuity_agent.print_response(
    "I will deposit $500 a year for 10 years, starting today, into an "
    "account earning 6% effective annual interest. What is that worth "
    "today?"
)

Because the question says "starting today," the agent should recognize an annuity-**due** and call
`annuity_due_pv(500, 0.06, 10)`, returning approximately **$3,900.85** — not the $3,680.04 an annuity-immediate
would give for the same ten payments made one year later.

### Compare: remove the timing cue

In [ ]:
annuity_agent.print_response(
    "I will deposit $500 a year for 10 years into an account earning "
    "6% effective annual interest. What is that worth today?"
)

Run both cells side by side. This is the case study's second demonstration — after Notebook 1's rate ambiguity
— of the same underlying lesson: a timing or convention detail invisible in casual language is not invisible to
the calculation, and a well-designed agent should surface the ambiguity (by asking) rather than silently resolve
it.

## 4. Reliability evaluation (Section 3.7)

In [ ]:
from agno.eval.reliability import ReliabilityEval

response = annuity_agent.run(
    "I will deposit $500 a year for 10 years, starting today, into an "
    "account earning 6% effective annual interest. What is that worth "
    "today?"
)

ReliabilityEval(
    name="Annuity Agent: correct tool for a due annuity",
    agent_response=response,
    expected_tool_calls=["annuity_due_pv"],
    expected_tool_call_arguments={
        "annuity_due_pv": {"payment": 500, "rate_per_period": 0.06, "n": 10},
    },
).run(print_results=True).assert_passed()

This is exactly the failure mode Section 3.7 highlights: an agent that calls `annuity_immediate_pv` here instead
of `annuity_due_pv` would still produce *a* numeric answer — just the wrong one for the timing described — and a
check on the final number alone, without checking which tool was used, would not necessarily catch it.

## Next

Continue with **Notebook 3 (Loan Amortization Agent, Section 4.3)** or **Notebook 4 (Multi-Agent Team,
Section 5)**.